### DimUser

In [0]:
import os
import sys
projec_path_var=(os.path.join(os.getcwd(),'..','..'))
sys.path.append(projec_path_var)
from pyspark.sql.functions import *
from pyspark.sql.types import *
from utils.reusable import Reusable

In [0]:
df=spark.read.format('parquet').load('abfss://bronze@storagespotifyproject2.dfs.core.windows.net/DimUser')

In [0]:
display(df)

## AutoLoader

DimUser

In [0]:
df_user=spark.readStream.format('cloudFiles').option('cloudFiles.format','parquet').option('cloudFiles.schemaLocation','abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimUser/checkpoint').load('abfss://bronze@storagespotifyproject2.dfs.core.windows.net/DimUser')

In [0]:
display(
    df_user,
    checkpointLocation="abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimUser/display_checkpoint"
)

In [0]:
df_user = df_user.withColumn('user_name', upper(col("user_name")))
df_user=df_user.dropDuplicates(["user_id"])
query = df_user.writeStream \
    .outputMode('append') \
    .format('delta') \
    .option('checkpointLocation', 'abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimUser/write_checkpoint_v2') \
    .trigger(availableNow=True) \
    .toTable('dim_user_silver')
query.awaitTermination()
display(spark.table('dim_user_silver'))

In [0]:
df_user.writeStream.format("delta").outputMode("append").option("checkpointLocation", "abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimUser/checkpointv3").trigger(once=True)\
    .option('path',"abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimUser/data").toTable('databricksazurespotify.silver.dim_user')

### Dim Artist

In [0]:
df_artist=spark.readStream.format('cloudFiles').option('cloudFiles.format','parquet').option('cloudFiles.schemaLocation','abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimArtist/Checkpoint').option('schemaEvolutionMode','addNewColumns').load('abfss://bronze@storagespotifyproject2.dfs.core.windows.net/DimArtist')
df_obj_reusable=Reusable()
df_artist=df_obj_reusable.dropCol(df_artist,*['_rescued_data'])


In [0]:
query=df_artist.writeStream.format('delta').outputMode('append').option("checkpointLocation", "abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimArtist/checkpointv2").trigger(once=True).toTable('dim_artist_silver')
query.awaitTermination()
display(spark.table('dim_artist_silver'))

In [0]:
df_artist=df_artist.dropDuplicates(["artist_id"])


In [0]:
df_artist.writeStream.format('delta').outputMode('append').option('CheckpointLocation','abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimArtist/checkpointv3').trigger(once=True).option('path',"abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimArtist/data").toTable('databricksazurespotify.silver.dim_artist')

DimTrack

In [0]:
df_track=spark.readStream.format('cloudFiles').option('cloudFiles.format','parquet').option('cloudFiles.schemaLocation','abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimTrack/checkpoint').option('schemaEvolutionMode','addNewColumns').load('abfss://bronze@storagespotifyproject2.dfs.core.windows.net/DimTrack')

In [0]:
df_track=df_track.withColumn('durationFlag',when(col('duration_sec')<160,'low').when((col('duration_sec')>160) & (col('duration_sec')<360),'medium').otherwise('high'))

In [0]:
query=df_track.writeStream.format('delta').outputMode('append').option('CheckpointLocation','abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimTrack/checkpointv2').trigger(once=True).toTable('dim_track_silver')
query.awaitTermination()
display(spark.table('dim_track_silver'))

In [0]:
df_track=df_track.withColumn('track_name',regexp_replace(col('track_name'),'-',' '))
df_ytack=df_track.dropDuplicates(["track_id"])

In [0]:
df_track.writeStream.format('delta').outputMode('append').option('CheckpointLocation','abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimTrack/checkpoint').trigger(once=True).option('path','abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimTrack/data').toTable('databricksazurespotify.silver.dim_track')

DimDate

In [0]:
df_date=spark.readStream.format('cloudFiles').option('cloudFiles.format','parquet').option('cloudFiles.schemaLocation','abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimDate/checkpoint').load('abfss://bronze@storagespotifyproject2.dfs.core.windows.net/DimDate')

In [0]:
df_date.writeStream.format('delta').outputMode('append').option('CheckpointLocation','abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimDate/checkpoint').trigger(once=True).option('path','abfss://silver@storagespotifyproject2.dfs.core.windows.net/DimDate/data').toTable('databricksazurespotify.silver.dim_date')

FactStream

In [0]:
df_factSteam=spark.readStream.format('cloudFiles').option('cloudFiles.format','parquet').option('cloudFiles.schemaLocation','abfss://silver@storagespotifyproject2.dfs.core.windows.net/FactStream/checkpoint').load('abfss://bronze@storagespotifyproject2.dfs.core.windows.net/FactStream')

In [0]:
df_factSteam.writeStream.format('delta').outputMode('append').option('CheckpointLocation','abfss://silver@storagespotifyproject2.dfs.core.windows.net/FactStream/checkpoint').trigger(once=True).option('path','abfss://silver@storagespotifyproject2.dfs.core.windows.net/FactStream/data').toTable('databricksazurespotify.silver.fact_stream')

In [0]:
%sql
SELECT * FROM databricksazurespotify.gold.dim_track